[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simon-clematide/colab-notebooks-for-teaching/blob/main/l06_cnn_companion.ipynb)

# L06 Companion: Convolutional Neural Networks for Text

This notebook accompanies the lecture **Convolutional Neural Networks for Text** (`cnn.tex`) and its bridge preview (`cnn-bridges.tex`).

### Learning Objectives & Overview
1. **Verify Slide Mechanics (Core):** Reproduce the exact 1D convolution arithmetic (`cnn:18`), hand calculation (`cnn:19`), parameter count activities (`cnn:15`), feature map information loss (`cnn:22`), and gradients through shared filters and max pooling (`cnn:26`, `cnn:26b`).
2. **Handle Variable Lengths & Padding (Core):** Discover the **padding trap** in text CNNs and implement correct masked max-over-time pooling (`cnn:27a`).
3. **Train & Compare on Language Identification (Core):** Train a character-level 1D CNN on a 6-language dataset and run a fair, controlled comparison against:
   - Baseline 1: Sparse character $n$-gram Linear Classifier (from L02).
   - Baseline 2: Continuous Bag-of-Characters / Mean Character Embedding baseline (L05).
4. **Inspect Learned Filters (Core):** Execute the empirical filter inspection protocol (`cnn:21`): extract top-activating character windows and correlate them with the classifier head weights.
5. **Explore Architectural Ablations & Transformer Bridges (Explore):**
   - Max vs. Mean pooling over convolutional features (`cnn:37`).
   - Multi-kernel widths (Kim 2014, `cnn:36`).
   - Width-1 Conv1D $\equiv$ Position-wise Linear layer (`cnnbridge:05`).
   - Token-wise Layer Normalization from scratch (`cnn:15a`).
   - Softmax competition in local attention (`cnnbridge:04`).


In [1]:
# Standard imports
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(42)
np.random.seed(42)
print("PyTorch Version:", torch.__version__)


PyTorch Version: 2.14.1


## 1. The Mechanics of 1D Convolution and Pooling *(Core)*

In deep learning libraries, **1D convolution** computes the unflipped cross-correlation between a sequence of token/character vectors and a sliding filter bank.

Let's begin by reproducing the small numerical examples from the lecture slides.


### 1.1 Worked Numerical Example (`cnn:18`)
On slide `cnn:18`, we have:
- A 1D sequence of length $n=4$ with $d=1$ input channel: $(1, 2, -1, 3)$.
- A kernel of width $k=2$: $(1, -1)$ with bias $b=0$.
- Unflipped cross-correlation formula for window $(x_1, x_2)$:
  $$z_i = 1 \cdot x_1 + (-1) \cdot x_2 + 0$$
- Valid windows:
  1. Window 1: $(1, 2) \implies z_1 = 1 - 2 = -1 \implies p_1 = \max(0, -1) = 0$
  2. Window 2: $(2, -1) \implies z_2 = 2 - (-1) = 3 \implies p_2 = \max(0, 3) = 3$
  3. Window 3: $(-1, 3) \implies z_3 = -1 - 3 = -4 \implies p_3 = \max(0, -4) = 0$
- Feature map $P = (0, 3, 0)$.
- Max-over-time pooling: $c = \max(0, 3, 0) = 3$.

Let's verify this using PyTorch's `nn.Conv1d`.


In [2]:
# Input shape for Conv1D in PyTorch: (batch_size, in_channels, sequence_length)
x = torch.tensor([[[1.0, 2.0, -1.0, 3.0]]]) # shape: (1, 1, 4)

# Define Conv1D: in_channels=1, out_channels=1, kernel_size=2, bias=False
conv = nn.Conv1d(in_channels=1, out_channels=1, kernel_size=2, bias=False)

# Manually assign the filter weights: (out_channels, in_channels, kernel_size)
with torch.no_grad():
    conv.weight.copy_(torch.tensor([[[1.0, -1.0]]]))

# Forward pass
z = conv(x)              # Linear scores
p = F.relu(z)            # Activation
c = p.amax(dim=2)        # Max-over-time pooling

print("Linear scores z: ", z.squeeze().detach().numpy())
print("Activated p:     ", p.squeeze().detach().numpy())
print("Pooled scalar c: ", c.item())

assert np.allclose(z.squeeze().detach().numpy(), [-1.0, 3.0, -4.0])
assert np.allclose(p.squeeze().detach().numpy(), [0.0, 3.0, 0.0])
assert c.item() == 3.0
print("✓ Slide cnn:18 exact arithmetic verified!")


Linear scores z:  [-1.  3. -4.]
Activated p:      [0. 3. 0.]
Pooled scalar c:  3.0
✓ Slide cnn:18 exact arithmetic verified!


### 1.2 Your Turn: Hand Calculation Check (`cnn:19`)
Slide `cnn:19` asks you to calculate:
- Input: $(2, -1, 3, 0)$
- Filter: $(-1, 1)$, bias $0$.
- Stride 1, no padding, ReLU activation, then max-over-time pooling.

*Self-check:*
1. Window 1: $(2, -1) \implies -1(2) + 1(-1) = -3 \to \text{ReLU} = 0$
2. Window 2: $(-1, 3) \implies -1(-1) + 1(3) = 4 \to \text{ReLU} = 4$
3. Window 3: $(3, 0) \implies -1(3) + 1(0) = -3 \to \text{ReLU} = 0$
- Pooled maximum: $4$, originating from the second window $(-1, 3)$.


In [3]:
x_yt = torch.tensor([[[2.0, -1.0, 3.0, 0.0]]])
with torch.no_grad():
    conv.weight.copy_(torch.tensor([[[-1.0, 1.0]]]))

z_yt = conv(x_yt)
p_yt = F.relu(z_yt)
c_yt = p_yt.amax(dim=2)

print("Linear scores: ", z_yt.squeeze().detach().numpy())
print("Activated:     ", p_yt.squeeze().detach().numpy())
print("Pooled scalar: ", c_yt.item())

assert np.allclose(z_yt.squeeze().detach().numpy(), [-3.0, 4.0, -3.0])
assert np.allclose(p_yt.squeeze().detach().numpy(), [0.0, 4.0, 0.0])
assert c_yt.item() == 4.0
print("✓ Slide cnn:19 Your Turn check verified!")


Linear scores:  [-3.  4. -3.]
Activated:      [0. 4. 0.]
Pooled scalar:  4.0
✓ Slide cnn:19 Your Turn check verified!


### 1.3 Shapes and Parameter Counts (`cnn:15`)
On slide `cnn:15`, we consider a realistic text convolutional layer:
- Input sequence: $n=50$ tokens.
- Embedding dimension: $d=300$.
- Kernel width: $k=3$ tokens per window.
- Number of filters: $m=100$ filters.
- Stride 1, no padding.

**Formulas:**
- Valid output length: $L = n - k + 1 = 50 - 3 + 1 = 48$.
- Flattened window dimension: $kd = 3 \times 300 = 900$.
- Weight matrix $U \in \mathbb{R}^{kd \times m} = \mathbb{R}^{900 \times 100}$.
- Bias $\mathbf{b} \in \mathbb{R}^{100}$.
- Total Conv1D parameters: $m \cdot kd + m = 100 \cdot 900 + 100 = 90{,}100$.

Compare this with a dense layer applied to the flattened $50$-token sentence:
- Flattened input: $n \cdot d = 50 \times 300 = 15{,}000$.
- Dense layer with 100 outputs: $15{,}000 \times 100 + 100 = 1{,}500{,}100$ parameters.

Locality and parameter sharing reduce the parameter count by **over 94%**!


In [4]:
n, d, k, m = 50, 300, 3, 100
conv_layer = nn.Conv1d(in_channels=d, out_channels=m, kernel_size=k, bias=True)

# Weight shape in PyTorch: (out_channels, in_channels, kernel_size)
print("Conv weight shape:", conv_layer.weight.shape) # (100, 300, 3)
print("Conv bias shape:  ", conv_layer.bias.shape)   # (100,)

num_conv_params = sum(p.numel() for p in conv_layer.parameters())
dense_flattened_params = (n * d) * m + m

print(f"Conv1D Parameters:          {num_conv_params:,}")
print(f"Flattened Dense Parameters: {dense_flattened_params:,}")

assert num_conv_params == 90_100
assert dense_flattened_params == 1_500_100

# Test tensor shape forward pass:
dummy_input = torch.randn(1, d, n) # (B=1, d=300, n=50)
out_p = conv_layer(dummy_input)
print("Output feature map shape:", out_p.shape)
assert out_p.shape == (1, m, 48)
print("✓ Slide cnn:15 shapes and parameter counts verified!")


Conv weight shape: torch.Size([100, 300, 3])
Conv bias shape:   torch.Size([100])
Conv1D Parameters:          90,100
Flattened Dense Parameters: 1,500,100
Output feature map shape: torch.Size([1, 100, 48])
✓ Slide cnn:15 shapes and parameter counts verified!


### 1.4 What Max Pooling Loses (`cnn:22`)
Slide `cnn:22` tests what survives and what disappears after max-over-time pooling.
Consider two feature maps $P_A$ and $P_B$ across 3 window positions and 2 filters:
$$P_A = \begin{pmatrix} 4 & 0 \\ 0 & 3 \\ 0 & 0 \end{pmatrix}, \qquad P_B = \begin{pmatrix} 0 & 3 \\ 4 & 0 \\ 4 & 0 \end{pmatrix}$$

Let's compute their max-pooled representations.


In [5]:
P_A = torch.tensor([[4.0, 0.0],
                    [0.0, 3.0],
                    [0.0, 0.0]]) # (3 windows, 2 filters)

P_B = torch.tensor([[0.0, 3.0],
                    [4.0, 0.0],
                    [4.0, 0.0]])

# Max-over-time pooling down the window axis (dim=0)
c_A = P_A.amax(dim=0)
c_B = P_B.amax(dim=0)

print("c_A:", c_A.numpy())
print("c_B:", c_B.numpy())

assert torch.equal(c_A, c_B)
print("✓ Slide cnn:22 verified: both feature maps pool to identical vectors (4.0, 3.0)!")


c_A: [4. 3.]
c_B: [4. 3.]
✓ Slide cnn:22 verified: both feature maps pool to identical vectors (4.0, 3.0)!


**Takeaway:**
- Max pooling keeps *whether* and *how strongly* each filter responded somewhere, not *where*: the pooled vector does not depend on the order of the feature-map rows.
- It discards:
  1. **Order of occurrence:** In $P_A$, Filter 1 fired before Filter 2. In $P_B$, Filter 2 fired first.
  2. **Frequency / Count:** In $P_B$, Filter 1 fired twice. In $P_A$, it fired only once.
- This is not invariance to arbitrary token order: reordering tokens changes the windows and thus the feature map itself.


### 1.5 Gradients Through Shared Filters and Max Pooling (`cnn:26`, `cnn:26b`)
Slide `cnn:26` follows one shared filter $\mathbf{u}=\mathbf{u}_1\oplus\mathbf{u}_2$ over two windows of three word vectors. Each use of $\mathbf{u}_1$ sends a contribution, and autograd **adds** them:
$$\frac{\partial\ell}{\partial\mathbf{u}_1}=\bar z_1\mathbf{w}_1+\bar z_2\mathbf{w}_2.$$
We use the values of the lecture demo and the probe loss $\ell = 2z_1 - z_2$, which supplies the incoming gradients $(\bar z_1, \bar z_2) = (2, -1)$.


In [6]:
# Columns are the word vectors w1, w2, w3 (d=3); Conv1d expects (batch, channels, positions)
W = torch.tensor([[1.0, 0.0, 2.0],
                  [0.0, 1.0, 3.0],
                  [2.0, 1.0, 0.0]]).T.unsqueeze(0)          # (1, 3, 3)

conv_shared = nn.Conv1d(in_channels=3, out_channels=1, kernel_size=2, bias=False)
with torch.no_grad():
    conv_shared.weight[0, :, 0] = torch.tensor([1.0, -1.0, 0.0])  # u1: applied to the first token of each window
    conv_shared.weight[0, :, 1] = torch.tensor([0.0, 1.0, 1.0])   # u2: applied to the second token

z = conv_shared(W).squeeze()        # scores (z1, z2)
loss = 2 * z[0] - z[1]              # probe loss: incoming gradients (2, -1)
loss.backward()

grad_u1 = conv_shared.weight.grad[0, :, 0]
grad_u2 = conv_shared.weight.grad[0, :, 1]
print("Scores z:         ", z.detach().numpy())
print("dl/du1 = g1 + g2: ", grad_u1.numpy(), "=", (2 * W[0, :, 0]).numpy(), "+", (-1 * W[0, :, 1] + 0.0).numpy())
print("dl/du2:           ", grad_u2.numpy())

assert torch.equal(z.detach(), torch.tensor([5.0, 0.0]))
assert torch.equal(grad_u1, torch.tensor([2.0, -1.0, 1.0]))
assert torch.equal(grad_u2, torch.tensor([-2.0, 1.0, 6.0]))
print("✓ Slide cnn:26 verified: the shared filter collects one contribution per window!")


Scores z:          [5. 0.]
dl/du1 = g1 + g2:  [ 2. -1.  1.] = [2. 0. 4.] + [ 0. -1. -3.]
dl/du2:            [-2.  1.  6.]
✓ Slide cnn:26 verified: the shared filter collects one contribution per window!


Slide `cnn:26b` adds max pooling: only the **winning** window receives the incoming gradient $\bar c$; the others receive zero.


In [7]:
p_act = torch.tensor([1.0, 4.0, 2.0], requires_grad=True)   # one filter's activations
c = p_act.max()
c.backward()                                                # incoming gradient c_bar = 1

print("Pooled c:  ", c.item())
print("dl/dp:     ", p_act.grad.numpy())
assert torch.equal(p_act.grad, torch.tensor([0.0, 1.0, 0.0]))
print("✓ Slide cnn:26b verified: max pooling routes the gradient to the winning window only!")


Pooled c:   4.0
dl/dp:      [0. 1. 0.]
✓ Slide cnn:26b verified: max pooling routes the gradient to the winning window only!


## 2. Data Pipeline & The Padding Trap (`cnn:27a`) *(Core)*

We now turn to a real text classification task: **Language Identification**.

### 2.1 The Dataset
We reuse the corpus of Exercise 1: WiLI-2018 Wikipedia paragraphs, 500 training and 500 test paragraphs per language. To make local character order matter, we pick six **closely related Romance languages** and classify **60-character snippets** instead of whole paragraphs:
- `spa` (Spanish), `por` (Portuguese), `glg` (Galician), `cat` (Catalan), `ita` (Italian), `fra` (French).

Of the 500 training paragraphs per language, 400 are used for training and 100 for the dev set, which selects the training epoch. The official test paragraphs are used only for the final comparison, so no snippet appears on both sides.


In [8]:
import os, urllib.request

# Exercise 1 data (WiLI-2018), downloaded once and cached (~130 MB in total)
FILES = {
    "x_train.txt": "1QP6YuwdKFNUPpvhOaAcvv2Pcp4JMbIRs",
    "x_test.txt": "1QVo7PZAdiZKzifK8kwhEr_umosiDCUx6",
    "y_train.txt": "1QbBeKcmG2ZyAEFB3AKGTgSWQ1YEMn2jl",
    "y_test.txt": "1QaZj6bI7_78ymnN8IpSk4gVvg-C9fA6X",
}
DATA_DIR = "wili_data"
os.makedirs(DATA_DIR, exist_ok=True)
for name, file_id in FILES.items():
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
        urllib.request.urlretrieve(url, path)

def read_lines(name):
    with open(os.path.join(DATA_DIR, name), encoding="utf-8") as f:
        return f.read().splitlines()

languages = ["cat", "fra", "glg", "ita", "por", "spa"]
lang2idx = {lang: i for i, lang in enumerate(languages)}
idx2lang = {i: lang for lang, i in lang2idx.items()}

SNIPPET_LEN = 60
rng = np.random.default_rng(42)

def snippet(text):
    """A random window of SNIPPET_LEN characters from one paragraph."""
    text = " ".join(text.split())
    if len(text) <= SNIPPET_LEN:
        return text
    start = rng.integers(0, len(text) - SNIPPET_LEN + 1)
    return text[start:start + SNIPPET_LEN]

def load(x_name, y_name):
    return [(lang, snippet(text))
            for text, lang in zip(read_lines(x_name), read_lines(y_name))
            if lang in lang2idx]

wili_train = load("x_train.txt", "y_train.txt")
test_data = load("x_test.txt", "y_test.txt")

# Dev set: 100 of the 500 training paragraphs per language
train_data, dev_data = [], []
for lang in languages:
    items = [s for s in wili_train if s[0] == lang]
    items = [items[i] for i in rng.permutation(len(items))]
    dev_data += items[:100]
    train_data += items[100:]
train_data = [train_data[i] for i in rng.permutation(len(train_data))]

print(f"Train: {len(train_data)}, Dev: {len(dev_data)}, Test: {len(test_data)} snippets")
for lang, text in train_data[:4]:
    print(f"  {lang}: {text!r}")


Train: 2400, Dev: 600, Test: 3000 snippets
  glg: ". Así mesmo, boa parte da trama d' O Silmarillion enrédase c"
  glg: 'As minguadas tropas xacobitas foron derrotadas polos británi'
  cat: 'itars russes i un pas hostil destinat a empitjorar les relac'
  por: 'pareceu. Ao chegar a um prédio onde moravam outros integrant'


In [9]:
# The split is done in the previous cell; the vocabulary comes from training data only.
# Build Character Vocabulary from Training Data
char_counts = {}
for _, text in train_data:
    for c in text:
        char_counts[c] = char_counts.get(c, 0) + 1

# Special tokens
PAD_IDX = 0
UNK_IDX = 1
char2idx = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
for c in sorted(c for c, n in char_counts.items() if n >= 2):  # rare characters -> <UNK>
    char2idx[c] = len(char2idx)
idx2char = {i: c for c, i in char2idx.items()}

print(f"Character vocabulary size: {len(char2idx)} (including <PAD>=0, <UNK>=1)")


Character vocabulary size: 158 (including <PAD>=0, <UNK>=1)


### 2.2 The Padding Trap & Correct Masked Pooling (`cnn:27a`)

In variable-length batches, shorter sequences are padded with `<PAD>` tokens to match the longest sequence.

> **Warning (`cnn:27a`):**
> If you simply call `p.amax(dim=2)` over a padded batch, **the padding can win the maximum!**
> Why?
> 1. Even if the `<PAD>` token has a zero embedding vector, the convolutional filter has a **bias term $b$**.
> 2. A window made **entirely of `<PAD>`** therefore scores exactly $b$; after ReLU, $\max(0, b)$.
> 3. Windows that mix real characters with `<PAD>` see an incomplete input and can score anything.
> 4. If $b > 0$ and every real window of a short text scores below $b$ (for example, all negative, so ReLU gives zero), a padding window **wins the maximum**: the pooled feature then depends on how much padding the batch added, not on the text.

**The Solution:**
Invalid window positions must be masked to $-\infty$ before applying max pooling.
Furthermore, if a sequence is shorter than the kernel width ($n < k$), no valid window exists; we must handle this cleanly with a fallback.


In [10]:
def masked_max_over_time(p, lengths, kernel_size):
    """
    Performs max-over-time pooling exclusively over valid windows.
    Args:
        p: Feature map tensor of shape (B, m, L), where L = n - k + 1
        lengths: Actual sequence lengths tensor of shape (B,)
        kernel_size: Width k of the convolution kernel
    Returns:
        c: Pooled representation of shape (B, m)
    """
    B, m, L = p.shape
    # Number of valid window starts for each example: max(0, length - k + 1)
    valid_starts = (lengths - kernel_size + 1).clamp(min=0) # (B,)
    
    # Position indices along output feature map: (1, 1, L)
    pos = torch.arange(L, device=p.device).unsqueeze(0).unsqueeze(0)
    # Mask of shape (B, 1, L): True for valid positions, False for padding positions
    valid_mask = pos < valid_starts.unsqueeze(1).unsqueeze(2)
    
    # Fill invalid positions with -infinity
    p_masked = p.masked_fill(~valid_mask, -float('inf'))
    
    # Max-over-time pooling along the sequence length axis
    c = p_masked.amax(dim=2)
    
    # Fallback for sequences shorter than kernel (valid_starts == 0): set vector to 0.0
    has_valid_windows = (valid_starts > 0).unsqueeze(1)
    c = torch.where(has_valid_windows, c, torch.zeros_like(c))
    return c

# Demonstration of the Padding Trap:
conv_demo = nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3, bias=True)
with torch.no_grad():
    conv_demo.weight.fill_(-1.0) # negative weights: text will yield <= 0
    conv_demo.bias.fill_(2.0)     # positive bias: padding will produce 2.0!

# Single example: 3 real tokens of 0.5, padded to length 6 with zeros
x_padded = torch.tensor([[[0.5, 0.5, 0.5, 0.0, 0.0, 0.0]]]) # length = 3 real, 3 pad
lens_demo = torch.tensor([3])

z_demo = conv_demo(x_padded)
p_demo = F.relu(z_demo)

naive_pool = p_demo.amax(dim=2)
correct_pool = masked_max_over_time(p_demo, lens_demo, kernel_size=3)

print("Activations across all windows (including pad):", p_demo.squeeze().detach().numpy())
print("Naive pooling (corrupted by pad bias):         ", naive_pool.item())
print("Correct masked pooling:                        ", correct_pool.item())

assert naive_pool.item() == 2.0  # Pad won!
assert correct_pool.item() == 0.5 # Real text window won!
print("✓ Padding trap demonstrated and solved with masked_max_over_time!")


Activations across all windows (including pad): [0.5 1.  1.5 2. ]
Naive pooling (corrupted by pad bias):          2.0
Correct masked pooling:                         0.5
✓ Padding trap demonstrated and solved with masked_max_over_time!


In [11]:
# PyTorch DataLoader with Batch Collation
def collate_fn(batch):
    labels, texts = zip(*batch)
    lengths = torch.tensor([len(t) for t in texts], dtype=torch.long)
    max_len = max(lengths).item()
    
    # Map chars to IDs with padding
    batch_ids = []
    for t in texts:
        ids = [char2idx.get(c, UNK_IDX) for c in t]
        padded = ids + [PAD_IDX] * (max_len - len(ids))
        batch_ids.append(padded)
        
    token_ids = torch.tensor(batch_ids, dtype=torch.long)
    label_ids = torch.tensor([lang2idx[l] for l in labels], dtype=torch.long)
    return token_ids, lengths, label_ids

# Prepare DataLoaders
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_data, batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, collate_fn=collate_fn)


## 3. Model Architectures & Controlled Training *(Core)*

We now implement and evaluate three distinct models under strictly matched conditions:
1. **Baseline 1 (L02):** Sparse Character $N$-Gram Logistic Regression.
2. **Baseline 2 (L05):** Continuous Bag of Characters (CBOC / Mean Embeddings).
3. **Model 3 (L06):** 1D Text CNN (`cnn:27`).


### 3.1 Baseline 1: Sparse Character N-gram Classifier (from L02)
We extract character 1- to 4-grams and train a standard logistic regression classifier. A second run with character **unigrams only** is a bag of characters: it sees which characters occur, but not their order.


In [12]:
X_train_raw = [t for _, t in train_data]
y_train_raw = [lang2idx[l] for l, _ in train_data]

X_test_raw = [t for _, t in test_data]
y_test_raw = [lang2idx[l] for l, _ in test_data]

def sparse_baseline(ngram_range):
    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=ngram_range, min_df=2)
    X_train_vec = vectorizer.fit_transform(X_train_raw)
    X_test_vec = vectorizer.transform(X_test_raw)
    lr_model = LogisticRegression(C=1.0, max_iter=1000)
    lr_model.fit(X_train_vec, y_train_raw)
    return lr_model.score(X_test_vec, y_test_raw)

acc_baseline_sparse = sparse_baseline((1, 4))
acc_baseline_unigram = sparse_baseline((1, 1))
print(f"Baseline 1  (Char 1-4-gram Logistic Regression) Test Accuracy: {acc_baseline_sparse*100:.2f}%")
print(f"Baseline 1b (Char unigram Logistic Regression)  Test Accuracy: {acc_baseline_unigram*100:.2f}%")


Baseline 1  (Char 1-4-gram Logistic Regression) Test Accuracy: 91.47%
Baseline 1b (Char unigram Logistic Regression)  Test Accuracy: 65.73%


### 3.2 Baseline 2: Continuous Bag-of-Characters (Mean Embeddings, L05)
This baseline maps each character to a $d$-dimensional embedding and computes the **masked mean vector** of the sentence.


In [13]:
class CharMeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.head = nn.Linear(embed_dim, num_classes)
        
    def forward(self, token_ids, lengths):
        # token_ids: (B, n)
        x = self.embedding(token_ids) # (B, n, d)
        
        # Mask out pad embeddings before summing
        mask = (token_ids != PAD_IDX).unsqueeze(-1) # (B, n, 1)
        x_masked = x * mask
        sum_emb = x_masked.sum(dim=1) # (B, d)
        
        # Mean across valid tokens
        mean_emb = sum_emb / lengths.unsqueeze(1).clamp(min=1) # (B, d)
        logits = self.head(mean_emb) # (B, C)
        return logits

model_mean = CharMeanEmbeddingClassifier(
    vocab_size=len(char2idx),
    embed_dim=64,
    num_classes=len(languages)
)
print(model_mean)


CharMeanEmbeddingClassifier(
  (embedding): Embedding(158, 64, padding_idx=0)
  (head): Linear(in_features=64, out_features=6, bias=True)
)


### 3.3 Model 3: 1D Character CNN (`cnn:27`)
Following slide `cnn:27`, the forward pipeline is:
```python
x = embedding(token_ids)   # (B, n, d)
x = x.transpose(1, 2)      # (B, d, n)
p = conv(x).relu()         # (B, m, L)
c = masked_max_over_time(p, lengths, k) # (B, m)
logits = head(c)           # (B, C)
```


In [14]:
class TextCNN1D(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, kernel_size, num_classes):
        super().__init__()
        self.kernel_size = kernel_size
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.conv = nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=kernel_size)
        self.head = nn.Linear(num_filters, num_classes)
        
    def forward(self, token_ids, lengths):
        # 1. Embedding lookup: (B, n, d)
        x = self.embedding(token_ids)
        # 2. Transpose for Conv1D: (B, d, n)
        x = x.transpose(1, 2)
        # 3. 1D Convolution + Activation: (B, m, L)
        p = F.relu(self.conv(x))
        # 4. Masked Max-Over-Time: (B, m)
        c = masked_max_over_time(p, lengths, self.kernel_size)
        # 5. Prediction Head: (B, C)
        logits = self.head(c)
        return logits, p

model_cnn = TextCNN1D(
    vocab_size=len(char2idx),
    embed_dim=64,
    num_filters=64,
    kernel_size=4,
    num_classes=len(languages)
)
print(model_cnn)


TextCNN1D(
  (embedding): Embedding(158, 64, padding_idx=0)
  (conv): Conv1d(64, 64, kernel_size=(4,), stride=(1,))
  (head): Linear(in_features=64, out_features=6, bias=True)
)


### 3.4 Training Loop
All neural models use Adam ($\text{lr} = 10^{-3}$) and CrossEntropyLoss. Each model trains until its dev accuracy stops improving (20 epochs for the CNNs, more for the slower-learning mean-pooling models), and we keep the parameters of the epoch with the best **dev** accuracy. The test set is used only once, for the final comparison.


In [15]:
def train_model(model, train_loader, dev_loader, epochs=20, lr=1e-3, report_every=5):
    """Train for a fixed budget and restore the parameters with the best dev accuracy."""
    best_dev, best_epoch, best_state = -1.0, 0, None
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_correct, total_tokens = 0, 0, 0
        for token_ids, lengths, labels in train_loader:
            optimizer.zero_grad()
            out = model(token_ids, lengths)
            logits = out[0] if isinstance(out, tuple) else out
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * len(labels)
            total_correct += (logits.argmax(dim=-1) == labels).sum().item()
            total_tokens += len(labels)
            
        train_acc = total_correct / total_tokens
        
        # Validation
        model.eval()
        dev_correct, dev_tokens = 0, 0
        with torch.no_grad():
            for token_ids, lengths, labels in dev_loader:
                out = model(token_ids, lengths)
                logits = out[0] if isinstance(out, tuple) else out
                dev_correct += (logits.argmax(dim=-1) == labels).sum().item()
                dev_tokens += len(labels)
        dev_acc = dev_correct / dev_tokens
        if dev_acc > best_dev:
            best_dev, best_epoch = dev_acc, epoch
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if epoch % report_every == 0:
            print(f"Epoch {epoch:02d} | Train Acc: {train_acc*100:.1f}% | Dev Acc: {dev_acc*100:.1f}%")
    model.load_state_dict(best_state)
    print(f"Restored epoch {best_epoch} (best dev accuracy {best_dev*100:.1f}%)")

print("--- Training Baseline 2: Mean Character Embeddings ---")
train_model(model_mean, train_loader, dev_loader, epochs=60)  # cheap; needs more epochs to plateau

print("\n--- Training Model 3: 1D Character CNN ---")
train_model(model_cnn, train_loader, dev_loader)


--- Training Baseline 2: Mean Character Embeddings ---


Epoch 05 | Train Acc: 49.3% | Dev Acc: 50.3%


Epoch 10 | Train Acc: 56.5% | Dev Acc: 54.8%


Epoch 15 | Train Acc: 62.0% | Dev Acc: 58.5%


Epoch 20 | Train Acc: 65.2% | Dev Acc: 61.8%


...

Epoch 60 | Train Acc: 74.8% | Dev Acc: 70.2%
Restored epoch 57 (best dev accuracy 70.7%)

--- Training Model 3: 1D Character CNN ---


Epoch 05 | Train Acc: 89.0% | Dev Acc: 86.3%


Epoch 10 | Train Acc: 96.4% | Dev Acc: 88.0%


Epoch 15 | Train Acc: 99.3% | Dev Acc: 88.8%


Epoch 20 | Train Acc: 100.0% | Dev Acc: 88.5%
Restored epoch 16 (best dev accuracy 89.0%)


## 4. Empirical Evaluation & Controlled Comparisons *(Core)*

Let's evaluate all three models on the unseen test set.


In [16]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for token_ids, lengths, labels in test_loader:
            out = model(token_ids, lengths)
            logits = out[0] if isinstance(out, tuple) else out
            all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    acc = (np.array(all_preds) == np.array(all_targets)).mean()
    return acc, np.array(all_preds), np.array(all_targets)

acc_mean, preds_mean, targets = evaluate_model(model_mean, test_loader)
acc_cnn, preds_cnn, _ = evaluate_model(model_cnn, test_loader)

print("="*60)
print(f"Baseline 1  (Char 1-4-gram Logistic Regression):  {acc_baseline_sparse*100:.2f}%")
print(f"Baseline 1b (Char unigram Logistic Regression):   {acc_baseline_unigram*100:.2f}%")
print(f"Baseline 2  (Mean of character embeddings):       {acc_mean*100:.2f}%")
print(f"Model 3     (1D character CNN, width k=4):        {acc_cnn*100:.2f}%")
print("="*60)


Baseline 1  (Char 1-4-gram Logistic Regression):  91.47%
Baseline 1b (Char unigram Logistic Regression):   65.73%
Baseline 2  (Mean of character embeddings):       66.20%
Model 3     (1D character CNN, width k=4):        85.17%


### 4.1 Observations & Inductive Bias
Approximate test accuracies on 3,000 snippets (exact values vary slightly between runs):

| Model | Sees local character order? | Test accuracy |
| :--- | :--- | :--- |
| Character unigram logistic regression | no | ~66% |
| Mean of character embeddings | no | ~66% |
| 1D character CNN, $k=4$, max over time | yes, within 4 characters | ~86% |
| Character 1–4-gram logistic regression | yes, within 4 characters | ~91% |

1. **Order-blind models plateau.** The two models that see only which characters occur reach about the same accuracy. Spanish, Galician and Portuguese share most characters; what separates them are short sequences such as `ção`, `ón `, `ny`, `ll` or the article `o`/`el`.
2. **Local order brings the gain.** The CNN and the n-gram classifier both see windows of up to four characters, and both clearly outperform the order-blind models.
3. **The CNN does not beat the sparse n-gram baseline here.** With only 2,400 training snippets, explicit n-gram features with a linear classifier remain a strong competitor. The CNN learns its features instead of enumerating them, with far fewer parameters than the n-gram vocabulary, but that does not guarantee higher accuracy on a small dataset.


## 5. Model Inspection: What Do Learned Filters Detect? (`cnn:21`) *(Core)*

Slide `cnn:21` outlines an empirical inspection protocol:
1. Run the trained CNN over held-out test texts.
2. For each filter $j$, find the input windows that produce the strongest activations $p_{i, j}$.
3. Inspect whether high-scoring windows correspond to recognizable linguistic cues.
4. Compare with the classifier head weights $W_{\text{head}}[:, j]$ to verify which language the filter votes for.

**Caution:** the head weights are small and the top windows of one filter often come from several languages. A filter's "highest-voting language" is a hint about its role, not a label: the head combines many filters, and a single filter's contribution is rarely decisive.


In [17]:
def inspect_filters(model, data, char2idx, top_k=6, num_filters_to_show=5):
    model.eval()
    k = model.kernel_size
    
    # Collect all valid character windows across the test dataset and their activations
    filter_activations = {f_idx: [] for f_idx in range(model.conv.out_channels)}
    
    with torch.no_grad():
        for lang, text in data:
            if len(text) < k:
                continue
            # Extract character windows
            windows = [text[i:i+k] for i in range(len(text) - k + 1)]
            ids = torch.tensor([[char2idx.get(c, UNK_IDX) for c in text]], dtype=torch.long)
            lens = torch.tensor([len(text)])
            
            _, p = model(ids, lens) # p shape: (1, m, L)
            p_sq = p.squeeze(0).detach().cpu().numpy() # (m, L)
            
            for f_idx in range(model.conv.out_channels):
                for w_idx, w_text in enumerate(windows):
                    act = p_sq[f_idx, w_idx]
                    if act > 0.05: # only track non-zero activations
                        filter_activations[f_idx].append((act, w_text, lang))
                        
    # Find filters with strong selective activations
    sorted_filters = sorted(filter_activations.keys(), 
                            key=lambda f: len(filter_activations[f]), 
                            reverse=True)
    
    print(f"Top inspected filters (kernel size k={k}):\\n")
    for f_idx in sorted_filters[:num_filters_to_show]:
        acts = filter_activations[f_idx]
        if not acts:
            continue
        # Sort descending by activation
        acts.sort(key=lambda x: x[0], reverse=True)
        # Deduplicate windows while preserving rank
        seen_windows = set()
        unique_top = []
        for a, w, l in acts:
            if w not in seen_windows:
                seen_windows.add(w)
                unique_top.append((a, w, l))
            if len(unique_top) == top_k:
                break
                
        # Head weights for this filter across classes
        head_weights = model.head.weight[:, f_idx].detach().cpu().numpy()
        top_lang_idx = head_weights.argmax()
        top_lang = idx2lang[top_lang_idx]
        
        print(f"Filter #{f_idx:02d} | Highest Voting Language: '{top_lang}' (weight = {head_weights[top_lang_idx]:+.3f})")
        print("  Top Activating Windows:")
        for score, win, win_lang in unique_top:
            print(f"    • '{win}' (act={score:.2f}, text lang={win_lang})")
        print("  Head Weights across classes:", 
              {idx2lang[i]: round(float(head_weights[i]), 2) for i in range(len(languages))})
        print("-" * 65)

inspect_filters(model_cnn, test_data, char2idx, top_k=5, num_filters_to_show=4)


Top inspected filters (kernel size k=4):\n
Filter #12 | Highest Voting Language: 'glg' (weight = +0.170)
  Top Activating Windows:
    • 'orou' (act=11.97, text lang=ita)
    • 'otou' (act=11.65, text lang=por)
    • ' rou' (act=11.42, text lang=fra)
    • 'o ou' (act=11.42, text lang=por)
    • ' Rou' (act=11.28, text lang=fra)
  Head Weights across classes: {'cat': -0.24, 'fra': 0.07, 'glg': 0.17, 'ita': -0.06, 'por': 0.03, 'spa': -0.14}
-----------------------------------------------------------------
Filter #50 | Highest Voting Language: 'fra' (weight = +0.297)
  Top Activating Windows:
    • 'évou' (act=13.13, text lang=fra)
    • 'érou' (act=12.69, text lang=fra)
    • 'écou' (act=12.58, text lang=fra)
    • 'émoi' (act=12.30, text lang=fra)
    • 'ição' (act=12.15, text lang=por)
  Head Weights across classes: {'cat': -0.16, 'fra': 0.3, 'glg': -0.02, 'ita': -0.2, 'por': 0.14, 'spa': -0.25}
-----------------------------------------------------------------
Filter #35 | Highest Vot

### 5.1 Your Turn: Interpreting Filter Diversity
Look at the top-activating windows and head weights above, then answer:
1. Which filters respond to windows that clearly belong to one language? Which respond to windows from several languages?
2. Does the language with the largest head weight always match the language of the top windows? Why might it not?
3. Rerun with another seed. Do you find the same filters again?

*Expected outcome:* some filters align with orthographic cues of one language; many respond across languages and contribute only in combination. Filters are learned feature detectors, not hand-written rules (`cnn:21`).


## 6. Explore: Pooling Ablations & Architectural Variants *(Explore)*

### 6.1 Mean vs. Max Pooling on the Same Convolutional Feature Map (`cnn:37`)
Slide `cnn:37` emphasizes:
- **Baseline 2** compared mean-pooling of *raw embeddings* with max-pooling of *convolutional features*.
- Here, we test a different question: what happens if we use **mean-pooling instead of max-pooling on the *exact same* convolutional feature map**?


In [18]:
class TextCNNMeanPool(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, kernel_size, num_classes):
        super().__init__()
        self.kernel_size = kernel_size
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.conv = nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=kernel_size)
        self.head = nn.Linear(num_filters, num_classes)
        
    def forward(self, token_ids, lengths):
        x = self.embedding(token_ids).transpose(1, 2)
        p = F.relu(self.conv(x)) # (B, m, L)
        
        # Masked MEAN over time
        B, m, L = p.shape
        valid_starts = (lengths - self.kernel_size + 1).clamp(min=1)
        pos = torch.arange(L, device=p.device).unsqueeze(0).unsqueeze(0)
        valid_mask = pos < valid_starts.unsqueeze(1).unsqueeze(2)
        
        p_masked = p * valid_mask # zero out pad positions
        c = p_masked.sum(dim=2) / valid_starts.unsqueeze(1) # (B, m)
        return self.head(c)

model_cnn_mean = TextCNNMeanPool(
    vocab_size=len(char2idx),
    embed_dim=64,
    num_filters=64,
    kernel_size=4,
    num_classes=len(languages)
)

print("--- Training CNN with Mean-over-Time Pooling ---")
train_model(model_cnn_mean, train_loader, dev_loader, epochs=40)
acc_cnn_mean, _, _ = evaluate_model(model_cnn_mean, test_loader)

print(f"CNN with Max-over-Time:  {acc_cnn*100:.2f}%")
print(f"CNN with Mean-over-Time: {acc_cnn_mean*100:.2f}%")


--- Training CNN with Mean-over-Time Pooling ---


Epoch 05 | Train Acc: 66.0% | Dev Acc: 60.2%


Epoch 10 | Train Acc: 76.6% | Dev Acc: 71.0%


Epoch 15 | Train Acc: 82.1% | Dev Acc: 78.7%


Epoch 20 | Train Acc: 86.0% | Dev Acc: 81.5%


Epoch 25 | Train Acc: 88.0% | Dev Acc: 82.0%


Epoch 30 | Train Acc: 89.7% | Dev Acc: 82.8%


Epoch 35 | Train Acc: 91.1% | Dev Acc: 83.7%


Epoch 40 | Train Acc: 92.0% | Dev Acc: 84.2%
Restored epoch 39 (best dev accuracy 84.2%)


CNN with Max-over-Time:  85.17%
CNN with Mean-over-Time: 81.57%


### 6.2 Multi-Width Convolution (Kim 2014, `cnn:36`)
In Yoon Kim's classic (2014) architecture, filters with multiple kernel sizes (e.g. $k \in \{2, 3, 4, 5\}$) run in parallel, and their pooled vectors are concatenated:
$$\mathbf{c} = \mathbf{c}_{k=2} \oplus \mathbf{c}_{k=3} \oplus \mathbf{c}_{k=4} \oplus \mathbf{c}_{k=5}$$


In [19]:
class KimMultiWidthCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, filters_per_k, kernel_sizes, num_classes):
        super().__init__()
        self.kernel_sizes = kernel_sizes
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=filters_per_k, kernel_size=k)
            for k in kernel_sizes
        ])
        self.head = nn.Linear(filters_per_k * len(kernel_sizes), num_classes)
        
    def forward(self, token_ids, lengths):
        x = self.embedding(token_ids).transpose(1, 2) # (B, d, n)
        pooled_outputs = []
        for conv, k in zip(self.convs, self.kernel_sizes):
            p = F.relu(conv(x))
            c = masked_max_over_time(p, lengths, kernel_size=k)
            pooled_outputs.append(c)
            
        c_all = torch.cat(pooled_outputs, dim=1) # concatenate across filter dimension
        logits = self.head(c_all)
        return logits

model_kim = KimMultiWidthCNN(
    vocab_size=len(char2idx),
    embed_dim=64,
    filters_per_k=24,
    kernel_sizes=[2, 3, 4, 5],
    num_classes=len(languages)
)

print(model_kim)
print("--- Training Multi-Width Kim CNN ---")
train_model(model_kim, train_loader, dev_loader)
acc_kim, _, _ = evaluate_model(model_kim, test_loader)
print(f"Kim Multi-Width CNN Test Accuracy: {acc_kim*100:.2f}%")


KimMultiWidthCNN(
  (embedding): Embedding(158, 64, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(64, 24, kernel_size=(2,), stride=(1,))
    (1): Conv1d(64, 24, kernel_size=(3,), stride=(1,))
    (2): Conv1d(64, 24, kernel_size=(4,), stride=(1,))
    (3): Conv1d(64, 24, kernel_size=(5,), stride=(1,))
  )
  (head): Linear(in_features=96, out_features=6, bias=True)
)
--- Training Multi-Width Kim CNN ---


Epoch 05 | Train Acc: 89.4% | Dev Acc: 86.2%


Epoch 10 | Train Acc: 96.6% | Dev Acc: 88.2%


Epoch 15 | Train Acc: 99.7% | Dev Acc: 89.8%


Epoch 20 | Train Acc: 100.0% | Dev Acc: 89.2%
Restored epoch 15 (best dev accuracy 89.8%)


Kim Multi-Width CNN Test Accuracy: 86.23%


## 7. Explore: Transformer Bridges: Channels vs. Positions *(Explore)*

This section provides executable foundations for L08 (*Transformers*), verifying the three core bridge concepts from `cnn-bridges.tex`.


### 7.1 Width-1 Conv1D $\equiv$ Position-Wise Linear Layer (`cnnbridge:05`)
Vaswani et al. (2017, Section 3.3) notes that the Transformer Feed-Forward Network (FFN) can be described as:
> *"two convolutions with kernel size 1"*

Let's prove this exact mathematical equivalence in PyTorch!


In [20]:
B_ex, n_ex, d_in, d_out = 2, 5, 16, 32

# Input tensor: (B, n, d)
X_tokens = torch.randn(B_ex, n_ex, d_in)

# 1. Position-wise Linear Layer
linear_layer = nn.Linear(in_features=d_in, out_features=d_out, bias=True)

# 2. Width-1 Conv1D Layer
conv1d_layer = nn.Conv1d(in_channels=d_in, out_channels=d_out, kernel_size=1, bias=True)

# Copy the exact same weights and biases
with torch.no_grad():
    # Linear weight is (d_out, d_in); Conv1D weight is (d_out, d_in, 1)
    conv1d_layer.weight.copy_(linear_layer.weight.unsqueeze(-1))
    conv1d_layer.bias.copy_(linear_layer.bias)

# Forward pass with Linear: directly acts on (B, n, d)
out_linear = linear_layer(X_tokens) # (B, n, d_out)

# Forward pass with Conv1D: requires transpose to (B, d, n), then transpose back
out_conv1d = conv1d_layer(X_tokens.transpose(1, 2)).transpose(1, 2) # (B, n, d_out)

# Check difference
max_diff = (out_linear - out_conv1d).abs().max().item()
print("Max absolute difference between Width-1 Conv1D and Position-Wise Linear:", max_diff)

assert np.isclose(max_diff, 0.0, atol=1e-6)
print("✓ Exact mathematical equivalence confirmed: Width-1 Conv1D == Position-Wise Linear!")


Max absolute difference between Width-1 Conv1D and Position-Wise Linear: 2.384185791015625e-07
✓ Exact mathematical equivalence confirmed: Width-1 Conv1D == Position-Wise Linear!


### 7.2 Token-Wise Layer Normalization from Scratch (`cnn:15a`)
On slide `cnn:15a`, we learn that **Layer Normalization** computes statistics across the $d$ feature dimensions of **each token separately**:
$$\operatorname{LN}(\mathbf{x}_i) = \boldsymbol{\gamma} \odot \frac{\mathbf{x}_i - \mu_i}{\sqrt{\sigma_i^2 + \epsilon}} + \boldsymbol{\beta}$$

Let's verify:
1. The slide calculation: $(1, 3) \to (-1, 1)$ and $(100, 104) \to (-1, 1)$.
2. **Token Independence:** Mutating token $j$ has **zero effect** on token $i$.


In [21]:
def token_layernorm(x, gamma=None, beta=None, eps=1e-5):
    """
    Token-wise Layer Normalization from scratch.
    x shape: (B, n, d)
    """
    mean = x.mean(dim=-1, keepdim=True)             # (B, n, 1)
    var = x.var(dim=-1, keepdim=True, unbiased=False) # (B, n, 1)
    x_hat = (x - mean) / torch.sqrt(var + eps)
    if gamma is not None and beta is not None:
        return gamma * x_hat + beta
    return x_hat

# Test 1: Verify slide arithmetic
toy_tokens = torch.tensor([[[1.0, 3.0], [100.0, 104.0]]]) # shape: (1, 2, 2)
normed_toy = token_layernorm(toy_tokens, eps=0.0)

print("Slide inputs:    (1, 3) and (100, 104)")
print("Normalized toy: ", normed_toy.squeeze().detach().numpy())
assert np.allclose(normed_toy.squeeze().detach().numpy(), [[-1.0, 1.0], [-1.0, 1.0]])
print("✓ Slide cnn:15a arithmetic verified!")

# Test 2: Token Independence Test
X_seq = torch.randn(2, 4, 8)
norm1 = token_layernorm(X_seq)

# Mutate token position 3 in sentence 0
X_seq_mutated = X_seq.clone()
X_seq_mutated[0, 3, :] += 50.0

norm2 = token_layernorm(X_seq_mutated)

# Token position 0 must be 100% identical!
token_0_diff = (norm1[0, 0, :] - norm2[0, 0, :]).abs().max().item()
print("Difference in token 0 output after mutating token 3:", token_0_diff)
assert token_0_diff == 0.0
print("✓ Token independence verified: LayerNorm never leaks information across positions!")


Slide inputs:    (1, 3) and (100, 104)
Normalized toy:  [[-1.  1.]
 [-1.  1.]]
✓ Slide cnn:15a arithmetic verified!
Difference in token 0 output after mutating token 3: 0.0
✓ Token independence verified: LayerNorm never leaks information across positions!


### 7.3 Softmax Competition in Local Attention (`cnnbridge:04`)
Slide `cnnbridge:04` poses a puzzle: token $A$ and token $B$ compete for attention.
$$e_A = 0, \quad e_B = 0 \implies \alpha_A = \frac{e^0}{e^0 + e^0} = \frac{1}{2}$$
If only $e_B$ changes to $\log 3$:
$$\alpha_A = \frac{e^0}{e^0 + e^{\log 3}} = \frac{1}{1 + 3} = \frac{1}{4}$$
The score between the center and token $A$ did not change, yet its normalized weight dropped from $0.5$ to $0.25$!


In [22]:
# Input 1: scores (0, 0)
e_input1 = torch.tensor([0.0, 0.0])
alpha_input1 = F.softmax(e_input1, dim=0)

# Input 2: scores (0, log(3))
e_input2 = torch.tensor([0.0, float(np.log(3))])
alpha_input2 = F.softmax(e_input2, dim=0)

print("Input 1 attention weights [alpha_A, alpha_B]:", alpha_input1.numpy())
print("Input 2 attention weights [alpha_A, alpha_B]:", alpha_input2.numpy())

assert np.allclose(alpha_input1.numpy(), [0.5, 0.5])
assert np.allclose(alpha_input2.numpy(), [0.25, 0.75])
print("✓ Slide cnnbridge:04 softmax competition verified!")


Input 1 attention weights [alpha_A, alpha_B]: [0.5 0.5]
Input 2 attention weights [alpha_A, alpha_B]: [0.25 0.75]
✓ Slide cnnbridge:04 softmax competition verified!


## Summary & Key Takeaways

1. **Locality & Weight Sharing:**
   - A 1D convolutional layer applies the same learned linear transformation to all sliding windows of width $k$.
   - This dramatically cuts parameter counts while maintaining sensitivity to local token/character arrangements.
2. **The Crucial Role of Masked Pooling:**
   - Unmasked max-over-time pooling on padded batches can let padding windows win the maximum, for example through a positive filter bias.
   - Pre-pooling $-\infty$ masking ensures only valid text windows contribute to sentence representations.
3. **Controlled Comparison:**
   - Among closely related languages, models that see local character order (CNN, character n-grams) clearly outperform order-blind ones (mean of embeddings, character unigrams).
   - On this small dataset, the sparse n-gram classifier remains as strong as the CNN: learned features do not automatically beat explicit ones.
4. **Bridge to Transformers:**
   - Width-1 convolution is identical to a position-wise dense layer (`nn.Linear`).
   - Layer Normalization operates strictly across features within a token, keeping token representations independent before cross-position mixing layers.
